# 2. M3c blacklist

Part of the **[Fig. 1 chapter](fig1.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import hicstraw
from glob import glob

import anndata
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import seaborn as sns
from ALLCools.clustering import *
from ALLCools.integration.seurat_class import SeuratIntegration
from ALLCools.plot import *
from sklearn.metrics import pairwise_distances, roc_auc_score
from sklearn.preprocessing import normalize

import warnings
warnings.filterwarnings("ignore")

mpl.style.use("default")
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = "Helvetica"


In [ ]:
indir = f'{ENTEX_ROOT}/'
outdir = f'{indir}analysis/m3ctest/'

In [ ]:
hiclist = np.sort(glob(f'{outdir}*.hic'))
print(len(hiclist))

In [ ]:
hiclist

In [ ]:
c = '2'
res = 10000
start, end = 88000000, 98000000


In [ ]:
plt.imshow(tmp, vmax=10)

In [ ]:
from scipy import ndimage as nd

dstall = []
for hicfile in hiclist:
    hic = hicstraw.HiCFile(hicfile)
    Q = hic.getMatrixZoomData(c, c, "observed", "NONE", "BP", res)
    tmp = Q.getRecordsAsMatrix(start, end, start, end)
    tmp = nd.rotate(tmp, 45, order=0, reshape=True, prefilter=False, cval=0)
    dstall.append(tmp)
    

In [ ]:
mpl.rcParams['svg.fonttype'] = 'none'

In [ ]:
resm = 1000
fig, axes = plt.subplots(len(hiclist), 1, figsize=(8, len(hiclist)), sharex='all', dpi=300)

for i in range(len(hiclist)):
    ax = axes[i]
    ct = hiclist[i].split('/')[-1].replace('.hic','')
    ax.set_title(ct)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    # vmax = np.percentile(tmp, 99.5)
    img = ax.imshow(dstall[i], cmap='afmhot_r', vmin=0, vmax=10,
                    interpolation='none', rasterized=True, aspect='auto')
    # sns.heatmap(dstall[i], cmap='afmhot_r', vmin=0, vmax=10, ax=ax, rasterized=True)
    h = len(dstall[i])
    ax.set_ylim([0.5*h, 0.4*h])
    ax.set_xlim([0, h])
    ax.set_yticks([])
    ax.set_yticklabels([])
    # ax.scatter((tmpl[:, 0]+tmpl[:, 1])/np.sqrt(2), 0.5*h-(tmpl[:, 1]-tmpl[:, 0])/np.sqrt(2), 
    #        alpha=0.1, s=10, marker='D', edgecolors='c', color='none')
    ax.set_xlim([0, (end-start-1)//res*np.sqrt(2)])
    ax.set_xticks(np.sqrt(2)*np.arange(0, end-start+1, 2e6)//res)
    ax.set_xticklabels([])
    ax.set_xticklabels([f'{(xx+start)/1e6}M' for xx in np.arange(0, end-start+1, 2e6)], rotation=0)
    
fig.tight_layout()
fig.savefig(f'{outdir}blacklist_example.svg', transparent=True)


In [ ]:
hicfile = hiclist[0]
hic = hicstraw.HiCFile(hicfile)
result = hic.getMatrixZoomData(c, c, "observed", "KR", "BP", res)

# result = hicstraw.straw('observed', 'NONE', hicfile, c, c, 'BP', res)

In [ ]:
plt.imshow(result.getRecordsAsMatrix(start,end,start,end), cmap='afmhot_r', vmin=0, vmax=20)